# Chapter 4 &mdash; Visitation Numbers, Pumping Up and Down

**Concept 13 of the Chapter 4 decomposition:** *Visitation Numbers, Pumping Up and Pumping Down*

Emboss a number at each visit. A twice-embossed state carries a pump you can skip or repeat.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Pumping-Up-And-Down/Concept-Pumping-Up-And-Down.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Imagine a rubber stamp **embossing a visitation number** on each state as you arrive.
Running a string of length $M \ge N$ through an $N$-state DFA, some state $s_p$ gets
stamped twice &mdash; at $v_p$ and $v_{p+k}$. **Any state embossed twice carries a pump**
of length $k>0$.

Two further journeys must then exist:

* **pumping down** &mdash; skip the loop;
* **pumping up** &mdash; take it again, and again.

The DFA is **forced** to admit all of them.

## 2. Definitions

### Emboss visitation numbers

In [ ]:
L3Z = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')

def emboss(D, s):
    q, stamps = D["q0"], [(0, D["q0"])]
    for i, ch in enumerate(s, 1):
        q = step_dfa(D, q, ch)
        stamps.append((i, q))
    return stamps

### Locate the first pump

In [ ]:
def first_pump(D, s):
    seen = {}
    q = D["q0"]; seen[q] = 0
    for i, ch in enumerate(s, 1):
        q = step_dfa(D, q, ch)
        if q in seen:
            return seen[q], i, q            # v_p, v_{p+k}, state
        seen[q] = i
    return None

## 3. Tests

The stamps, and the first repeat.

In [ ]:
w = '0100'
for i, q in emboss(L3Z, w): print("  v%-2d -> %s" % (i, q))
p, pk, st = first_pump(L3Z, w)
print("\nfirst pump: state %s stamped at v%d and v%d, so k = %d" % (st, p, pk, pk-p))
assert pk > p

Split accordingly, then pump **down** and **up**.

In [ ]:
p, pk, st = first_pump(L3Z, w)
x, y, z = w[:p], w[p:pk], w[pk:]
print("x=%r  y=%r  z=%r   (y non-empty: %s, |xy| = %d)" % (x, y, z, y != '', len(x+y)))
assert y != ''
print("\npump down (i=0):", repr(x + z),      "accepted?", accepts_dfa(L3Z, x + z))
for i in [1,2,3,5]:
    s = x + y*i + z
    print("pump up   (i=%d): %-14r accepted? %s" % (i, s, accepts_dfa(L3Z, s)))
assert all(accepts_dfa(L3Z, x + y*i + z) for i in range(8))

The machine cannot distinguish them, so it must accept them all.

In [ ]:
ends = {run_dfa(L3Z, x + y*i + z) for i in range(8)}
print("final state for every i :", ends)
assert len(ends) == 1, "all pumped strings end in the SAME state"

## 4. Exercises


1. Why do we focus on the **first** pump? What would change if we chose another?
2. Find a string with **two** distinct pumps. Does pumping either one work?
3. What is the largest $|xy|$ can be, and why does that matter later?

In [ ]:
# Your work for the exercises above.